# 10_train_descriptors — descriptor로 학습 + 물성 해석

**한 줄 요약:** descriptor로 모델을 학습해 성능을 보고, **어떤 물성이 활성과 연관되는지**(중요도 + active/inactive 평균 방향)를 해석한다.
**큰 흐름:** ① 준비·읽기 → ② 물질 단위 정리 → ③ 파이프라인 5-fold CV → ④ 전체 학습·중요도·저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + descriptor 데이터 읽기
라이브러리(전처리 파이프라인 포함)를 가져오고 09에서 만든 descriptor CSV를 읽는다.

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

SRC = "data/HSD17B13_descriptors.csv"
IMP_OUT = "data/HSD17B13_descriptor_importance.csv"
MODEL_OUT = "data/HSD17B13_descriptor_model.pkl"
META = ["canonical_smiles", "ic50_nM", "relation", "sources", "label"]

df = pd.read_csv(SRC)
df = df.dropna(subset=["label"]).copy()
desc_cols = [c for c in df.columns if c not in META]

🔎 **코드 뜯어보기 (셀 1)**
- `from sklearn.pipeline import Pipeline` : 전처리+모델을 **하나로 묶는** 도구. `from sklearn.impute import SimpleImputer` : 빈 값을 대푯값으로 채우는 도구.

### 셀 2 — 물질 단위로 정리
같은 분자는 descriptor가 같으니 한 줄로 합치고, 라벨은 하나라도 active면 active.

In [ ]:
agg = {c: "first" for c in desc_cols}
agg["label"] = "max"
comp = df.groupby("canonical_smiles").agg(agg).reset_index()
comp["label"] = comp["label"].astype(int)

X = comp[desc_cols].replace([np.inf, -np.inf], np.nan)
y = comp["label"].to_numpy()
print(f"물질 {len(comp)}개 | active {int(y.sum())} / inactive {int((y==0).sum())} "
      f"| descriptor {len(desc_cols)}종")

🔎 **코드 뜯어보기 (셀 2)**
- `agg = {c: "first" for c in desc_cols}` : **딕셔너리 컴프리헨션** — 모든 descriptor 열을 '첫 값'으로 집계하도록 지정. `agg["label"] = "max"` : 라벨만 최대(=active 우선). `df.groupby("...").agg(agg)`=그 규칙으로 물질별 합치기.

### 셀 3 — 파이프라인으로 5-fold 교차검증
결측 대체와 모델을 하나로 묶어(파이프라인) 교차검증 성능을 잰다.

In [ ]:
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(n_estimators=400, class_weight="balanced",
                           random_state=42, n_jobs=-1, verbosity=-1)),
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(pipe, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
pred = (proba >= 0.5).astype(int)
tn, fp_, fn, tp = confusion_matrix(y, pred).ravel()
print("\n=== descriptor 모델 5-fold CV ===")
print(f"ROC-AUC {roc_auc_score(y, proba):.3f} | PR-AUC {average_precision_score(y, proba):.3f} "
      f"| Recall(act) {tp/(tp+fn):.3f} | Prec(act) {tp/(tp+fp_):.3f}")
print("(참고: fingerprint 모델(실측데이터)은 ROC-AUC ~0.90)")

🔎 **코드 뜯어보기 (셀 3)**
- `Pipeline([("impute", SimpleImputer(strategy="median")), ("clf", LGBMClassifier(...))])` : **① 빈 값을 중앙값으로 채우고 → ② 모델 학습** 을 한 묶음으로. 교차검증 시 각 조각마다 이 과정이 통째로 적용돼 누수를 막는다.
- `cross_val_predict(pipe, X, y, cv=skf, method="predict_proba")` : 파이프라인째 교차검증 예측.

### 셀 4 — 전체 학습 → 중요 물성 + 방향 해석 → 저장
전체로 학습해 중요한 descriptor와, active/inactive에서 값이 높/낮은 방향까지 계산해 저장한다.

In [ ]:
pipe.fit(X, y)
imp = pipe.named_steps["clf"].feature_importances_   # gain 아님, split 기반 → gain으로 재계산
booster = pipe.named_steps["clf"].booster_
gain = booster.feature_importance(importance_type="gain")

Xf = pipe.named_steps["impute"].transform(X)
Xf = pd.DataFrame(Xf, columns=desc_cols)
act_mean = Xf[y == 1].mean()
ina_mean = Xf[y == 0].mean()
pooled = Xf.std().replace(0, np.nan)
cohen = (act_mean - ina_mean) / pooled          # 표준화 효과크기(방향+세기)

res = pd.DataFrame({
    "descriptor": desc_cols,
    "gain_importance": gain,
    "active_mean": act_mean.values,
    "inactive_mean": ina_mean.values,
    "std_effect": cohen.values,               # +면 active에서 높음
}).sort_values("gain_importance", ascending=False).reset_index(drop=True)
res.to_csv(IMP_OUT, index=False)

with open(MODEL_OUT, "wb") as f:
    pickle.dump({"pipeline": pipe, "desc_cols": desc_cols}, f)

print(f"\n=== 활성과 가장 연관된 물성(중요도) 상위 20 ===")
print(f"{'descriptor':22s} {'중요도':>10s} {'active평균':>11s} {'inact평균':>11s} {'방향':>6s}")
for _, r in res.head(20).iterrows():
    arrow = "↑active" if r.std_effect > 0 else "↓active"
    print(f"{r.descriptor:22s} {r.gain_importance:10.0f} "
          f"{r.active_mean:11.2f} {r.inactive_mean:11.2f} {arrow:>7s}")

print(f"\n저장: {IMP_OUT} | {MODEL_OUT}")
print("주의: descriptor끼리 상관 높으면 중요도가 서로 나뉘어 과소평가될 수 있음 → 방향(std_effect)과 함께 해석")

🔎 **코드 뜯어보기 (셀 4)**
- `pipe.fit(X, y)` : 전체 학습. `pipe.named_steps["clf"].booster_.feature_importance(importance_type="gain")` : 각 descriptor의 **중요도(gain)**.
- `act_mean = Xf[y == 1].mean()` : active 그룹의 평균. `cohen = (act_mean - ina_mean) / pooled` : **표준화 효과크기**(+면 active에서 큼, −면 작음) → 방향 해석.
- `for _, r in res.head(20).iterrows():` : 상위 20개를 한 줄씩 출력. `pickle.dump(...)` : 모델(파이프라인) 저장.